# RelCheck — Hallucination Screening (BLIP-2)

**Purpose:** Screen 100 R-Bench images to find BLIP-2 captions with relational hallucinations.

**GPU required** — BLIP-2 runs locally on Colab GPU. Llama judges via Together API.

**Output:** List of candidate images where BLIP-2 captions make false relational claims,
broken down by relation type (spatial, action, attribute).
These candidates go into the full RelCheck pipeline.


In [ ]:
# ============================================================
# Cell 0 — Setup (BLIP-2 on GPU + Llama via Together API)
# ============================================================
import os, json, time, random, re
from collections import defaultdict
from openai import OpenAI
from PIL import Image

TOGETHER_API_KEY = ""  # ← Paste your key here
client = OpenAI(base_url="https://api.together.xyz/v1", api_key=TOGETHER_API_KEY)

LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"  # Llama for R-POPE judging

# Load BLIP-2
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

print("Loading BLIP-2 (blip2-flan-t5-xl)...")
blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float16, device_map="auto"
)
print("✅ BLIP-2 loaded on GPU")
print("✅ Together API ready. Paste your key above.")


In [ ]:
# ============================================================
# Cell 1 — Load R-Bench Data + Select Images
# ============================================================
from google.colab import drive
from pathlib import Path

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

# Load R-Bench
with open(RBENCH_PATH) as f:
    rbench_data = json.load(f)
print(f"R-Bench: {len(rbench_data)} questions total")

# Build lookup by image
rbench_by_image = defaultdict(list)
for entry in rbench_data:
    rbench_by_image[entry['image']].append(entry)
print(f"R-Bench covers {len(rbench_by_image)} unique images")

# Find images that exist on Drive AND have R-Bench questions
cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + \
                list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
cached_stems = {p.stem: p for p in cached_images}

matched_images = {}
for rb_key in rbench_by_image:
    rb_clean = rb_key.replace('.jpg', '').replace('.jpeg', '').replace('.png', '')
    if rb_clean in cached_stems:
        matched_images[rb_clean] = {
            "path": str(cached_stems[rb_clean]),
            "questions": rbench_by_image[rb_key],
        }

print(f"Images on Drive with R-Bench questions: {len(matched_images)}")

# Sample N images for screening
N_SCREEN = 100
random.seed(42)
if len(matched_images) > N_SCREEN:
    selected_ids = random.sample(list(matched_images.keys()), N_SCREEN)
else:
    selected_ids = list(matched_images.keys())
    print(f"  (Using all {len(selected_ids)} available images)")

screen_images = {k: matched_images[k] for k in selected_ids}
total_questions = sum(len(v["questions"]) for v in screen_images.values())
print(f"\nScreening {len(screen_images)} images with {total_questions} R-Bench questions")



In [ ]:
# ============================================================
# Cell 2 — Caption with BLIP-2 (local GPU)
# ============================================================

def caption_with_blip2(image_path):
    """Generate caption via BLIP-2 on Colab GPU."""
    img = Image.open(image_path).convert("RGB")
    prompt = "Describe the relationships between objects and people in this image."
    inputs = blip2_processor(images=img, text=prompt, return_tensors="pt").to(
        blip2_model.device, torch.float16
    )
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80, temperature=0.7)
    caption = blip2_processor.decode(out[0], skip_special_tokens=True).strip()
    return caption

captions = {}
for i, (img_id, info) in enumerate(screen_images.items()):
    try:
        cap = caption_with_blip2(info["path"])
        captions[img_id] = cap
        if (i + 1) % 10 == 0 or i == 0:
            print(f"[{i+1}/{len(screen_images)}] {img_id[:20]}: {cap[:80]}...")
    except Exception as e:
        print(f"[{i+1}] {img_id[:20]}: FAILED — {e}")
        captions[img_id] = ""

print(f"\nCaptioned {sum(1 for c in captions.values() if c)}/{len(screen_images)} images")


In [ ]:
# ============================================================
# Cell 3 — R-POPE Screening (find hallucinated captions)
# ============================================================
# For each image: Llama answers R-Bench questions based on caption only.
# GT=no + LLM=yes → caption makes a FALSE relational claim (hallucination!)
# GT=yes + LLM=no → caption OMITS information (not fixable by RelCheck)

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""

def rpope_judge(caption, question):
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content":
                RPOPE_PROMPT.format(caption=caption, question=question)}],
            temperature=0.0, max_tokens=5,
        )
        answer = resp.choices[0].message.content.strip().lower()
        if "yes" in answer and "no" not in answer:
            return "yes"
        elif "no" in answer:
            return "no"
        return "unknown"
    except Exception as e:
        return "unknown"

# ── Classify relation type from question text ──
def classify_question(question):
    """Classify R-Bench question into relation subcategory."""
    q = question.lower()

    # Action verbs (check first — most specific)
    action_verbs = [
        "playing", "holding", "eating", "sitting", "standing", "running",
        "walking", "riding", "driving", "reading", "closing", "opening",
        "looking", "talking", "carrying", "throwing", "catching", "cooking",
        "swimming", "flying", "climbing", "jumping", "dancing", "sleeping",
        "drinking", "writing", "pointing", "pulling", "pushing", "cutting",
        "hitting", "kicking", "hugging", "kissing", "fighting", "leaning",
        "lying", "bending", "reaching", "touching", "feeding", "chasing",
        "surfing", "skiing", "skating", "smiling", "crying", "waving",
    ]
    for verb in action_verbs:
        if verb in q:
            return "ACTION"

    # Spatial prepositions (as whole words, more careful matching)
    spatial_phrases = [
        " on a ", " on the ", " on top ", " in a ", " in the ",
        " inside ", " outside ", " near ", " next to ", " behind ",
        " above ", " below ", " under ", " beside ", " between ",
        " in front of ", " across ", " along ", " around ",
    ]
    for phrase in spatial_phrases:
        if phrase in f" {q} ":
            return "SPATIAL"

    # Attribute patterns
    attribute_kw = [
        "wearing", "color", "colour", "red", "blue", "green", "yellow",
        "black", "white", "brown", "large", "small", "big", "tall", "short",
        "old", "young", "made of", "type of", "kind of",
    ]
    for kw in attribute_kw:
        if kw in q:
            return "ATTRIBUTE"

    return "OTHER"

# ── Run screening ──
results = {}
false_claims = []  # GT=no, LLM=yes (hallucinations!)
omissions = []     # GT=yes, LLM=no (missing info)
correct = []

total_q = 0
for i, (img_id, info) in enumerate(screen_images.items()):
    caption = captions.get(img_id, "")
    if not caption:
        continue

    img_results = {"caption": caption, "questions": [], "false_claims": 0, "omissions": 0, "correct": 0}

    for entry in info["questions"][:10]:
        question = entry["question"]
        gt = entry["answer"]
        llm_ans = rpope_judge(caption, question)
        rel_type = classify_question(question)
        total_q += 1

        q_result = {
            "question": question, "gt": gt, "llm_answer": llm_ans,
            "correct": llm_ans == gt, "rel_type": rel_type,
        }
        img_results["questions"].append(q_result)

        if llm_ans == gt:
            img_results["correct"] += 1
            correct.append({"img_id": img_id, **q_result})
        elif gt == "no" and llm_ans == "yes":
            # FALSE CLAIM — caption says something that isn't true
            img_results["false_claims"] += 1
            false_claims.append({"img_id": img_id, "caption": caption, **q_result})
        elif gt == "yes" and llm_ans == "no":
            # OMISSION — caption doesn't mention something that's there
            img_results["omissions"] += 1
            omissions.append({"img_id": img_id, "caption": caption, **q_result})

    results[img_id] = img_results

    if (i + 1) % 20 == 0:
        print(f"  Screened {i+1}/{len(screen_images)} images ({total_q} questions)...")
    time.sleep(0.2)

print(f"\n{'='*60}")
print(f"SCREENING RESULTS — {total_q} questions across {len(results)} images")
print(f"{'='*60}")
print(f"  Correct:      {len(correct)} ({len(correct)/total_q:.1%})")
print(f"  FALSE CLAIMS: {len(false_claims)} ({len(false_claims)/total_q:.1%})  ← hallucinations RelCheck can fix")
print(f"  Omissions:    {len(omissions)} ({len(omissions)/total_q:.1%})  ← missing info, not fixable")



In [ ]:
# ============================================================
# Cell 4 — Analyze Hallucinations by Relation Type
# ============================================================

print("=" * 60)
print("FALSE CLAIMS BY RELATION TYPE (these are fixable hallucinations)")
print("=" * 60)

# Group false claims by type
fc_by_type = defaultdict(list)
for fc in false_claims:
    fc_by_type[fc["rel_type"]].append(fc)

for rtype in ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]:
    items = fc_by_type.get(rtype, [])
    print(f"\n{'─'*50}")
    print(f"{rtype}: {len(items)} false claims")
    print(f"{'─'*50}")
    for fc in items[:10]:  # Show up to 10 examples per type
        print(f"  [{fc['img_id'][:15]}]")
        print(f"    Q: {fc['question']}")
        print(f"    GT: {fc['gt']} | Caption-based answer: {fc['llm_answer']}")
        print(f"    Caption: {fc['caption'][:100]}...")

# ── Summary table ──
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"{'Type':<12} {'False Claims':<15} {'Omissions':<12} {'Correct':<10}")
print(f"{'─'*50}")

om_by_type = defaultdict(list)
for om in omissions:
    om_by_type[om["rel_type"]].append(om)
cor_by_type = defaultdict(list)
for c in correct:
    cor_by_type[c["rel_type"]].append(c)

for rtype in ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]:
    fc_n = len(fc_by_type.get(rtype, []))
    om_n = len(om_by_type.get(rtype, []))
    cor_n = len(cor_by_type.get(rtype, []))
    total = fc_n + om_n + cor_n
    print(f"{rtype:<12} {fc_n:<15} {om_n:<12} {cor_n:<10}  (total: {total})")

# ── Images with most false claims (best candidates for RelCheck) ──
print(f"\n{'='*60}")
print(f"TOP HALLUCINATION CANDIDATES (most false claims per image)")
print(f"{'='*60}")

img_fc_counts = [(img_id, r["false_claims"], r["omissions"], len(r["questions"]))
                  for img_id, r in results.items() if r["false_claims"] > 0]
img_fc_counts.sort(key=lambda x: x[1], reverse=True)

print(f"\n{len(img_fc_counts)} images have at least 1 false claim")
print(f"\n{'Image ID':<20} {'False Claims':<15} {'Omissions':<12} {'Total Q':<10}")
print(f"{'─'*55}")
for img_id, fc, om, tq in img_fc_counts[:30]:
    print(f"{img_id[:20]:<20} {fc:<15} {om:<12} {tq:<10}")
    cap = results[img_id]["caption"]
    print(f"  Caption: {cap[:90]}...")
    for q in results[img_id]["questions"]:
        if not q["correct"] and q["gt"] == "no":
            print(f"  ❌ FALSE: {q['question']}")
            print(f"     (GT=no, caption implies yes → hallucination)")

# ── Save candidates for full RelCheck run ──
candidates = [img_id for img_id, fc, om, tq in img_fc_counts]
save_path = "/content/drive/MyDrive/RelCheck_Data/hallucination_candidates_blip2.json"
with open(save_path, "w") as f:
    json.dump({
        "candidates": candidates,
        "total_screened": len(results),
        "total_with_false_claims": len(candidates),
        "details": {img_id: results[img_id] for img_id in candidates},
    }, f, indent=2)
print(f"\n✅ Saved {len(candidates)} candidate images to {save_path}")
print(f"   Use these for the full RelCheck pipeline run.")
